In [42]:
import torch 
import torchvision
import torch.nn as nn 
import torch.optim as opt 
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

In [43]:
class Lenet(nn.Module): 
    def __init__(self): 
        super(Lenet, self).__init__()
        #only white or black for mnist so 1 and searches for 6 basic feats 
        self.convb1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool = nn.AvgPool2d(2,2)
        self.convb2= nn.Conv2d(6, 16, kernel_size=5)
        
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
    
    def forward(self, x): 
        #using the torch relu directly instead of sep var 
        x = self.pool(torch.relu(self.convb1(x)))
        x = self.pool(torch.relu(self.convb2(x)))
        #flattening again 16*5*5 to the batch size 
        x = x.view(x.size(0), -1)
        #same as the prev one apply relu to all layers except output layer
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x         

In [44]:
model = Lenet()
crit = nn.CrossEntropyLoss()
opt = opt.Adam(model.parameters(), lr = 0.001)

In [45]:
#process only 32x32 images dataset has 28x28
transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor()
])
train_d = torchvision.datasets.MNIST(root='./data', download=True, train=True, transform=transform)
test_d = torchvision.datasets.MNIST(root='./data', download=True, train= False ,transform=transform)

train_loader = DataLoader(train_d, batch_size=64, shuffle=True)
test_loader = DataLoader(train_d, batch_size=64, shuffle=False)

In [46]:
for e in range(5):
    r_loss = 0 
    
    for im, la in train_loader: 
        
        opt.zero_grad()
        
        op = model(im)
        loss = crit(op, la)
        
        loss.backward()
        opt.step()
        
        r_loss += loss.item()
    
    print(f"Epoch: {e+1}, Training loss: {r_loss/len(train_loader)}")        

Epoch: 1, Training loss: 0.33406687469314983
Epoch: 2, Training loss: 0.08886044084686222
Epoch: 3, Training loss: 0.06251614936850845
Epoch: 4, Training loss: 0.04875878230949093
Epoch: 5, Training loss: 0.039428206257594585


In [47]:
corr = 0 
tot = 0 

with torch.no_grad(): 
    for im, la in test_loader: 
        op = model(im) 
        _, pred = torch.max(op.data, 1)
        corr += (pred==la).sum().item()
        tot += la.size(0)

acc = 100* (corr/tot)
print(f"Test accuracy: {acc:.2f}")

Test accuracy: 98.95
